In [1]:
import numpy as np
from sklearn.model_selection import train_test_split

'''
These are all .py files that perform different functions and are imported into this main file to be used
'''

from pipelines.data_loader import load_all_subjects
from pipelines.feature_extraction import (
    create_model_windows,
    #create_raw_windows,

    build_logistic_features,
    build_svm_features,
    build_knn_features,
    build_rf_features,
    build_gb_features,
    build_xgb_features
)
from pipelines.evaluate import evaluate_model
from pipelines.logistics_pipeline import build_model as logistic_model
from pipelines.svm_pipeline import build_model as svm_model
from pipelines.randomforest_pipeline import build_model as rf_model
from pipelines.xgboost_pipeline import build_model as xgb_model
from pipelines.knn_pipeline import build_model as knn_model
from pipelines.gradientboost_pipeline import build_model as gb_model

from pipelines.train import train_models

DATA_DIR = 'data/WESAD'

models, signals, labels, X_models, y_models, model_feature_names, all_X, all_y = train_models(DATA_DIR)

#CHANGE DATA_DIR to 'data/WESAD' to look at how wesad works'
#ALSO I SENT THE AMIGOS DATASET AS AMIGOS/USER1/USER1 THEN THE CONTENTS - YOU WILL NEED TO MAKE IT AMIGOS/USER1 THEN THE LABEL FOLDER AND CSVS


data/WESAD\S10.pkl
data/WESAD\S11.pkl
data/WESAD\S13.pkl
data/WESAD\S14.pkl
data/WESAD\S15.pkl
data/WESAD\S16.pkl
data/WESAD\S17.pkl
data/WESAD\S2.pkl
data/WESAD\S3.pkl
data/WESAD\S4.pkl


UnpicklingError: invalid load key, '\x5c'.

In [ ]:
#Code for all the visualizations

import numpy as np
import pandas as pd

from sklearn.metrics import roc_curve, auc

import matplotlib.patches as mpatches

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    roc_curve,
    auc
)
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import learning_curve

from sklearn.tree import plot_tree

from xgboost import (
    plot_importance,
    plot_tree as xgb_plot_tree
)


SIGNAL_UNITS = {
    "EDA": "μS",
    "BVP": "nWatts",
    "TEMP": "°C ",
    "ACC" : "1/64g"
}

EMOTION_LABELS = [
    'Happy',
    'Sad',
    'Calm',
    'Angry'
]


def plot_feature_importance(
    model,
    feature_names,
    model_name,
    top_n=15
):

   

    actual_model = model

    if hasattr(model, 'named_steps'):

        if 'model' in model.named_steps:

            actual_model = model.named_steps['model']


    if hasattr(actual_model, 'feature_importances_'):

        importances = actual_model.feature_importances_

    else:

        print(
            f"{model_name} has no feature_importances_"
        )

        return

    num_importances = len(importances)
    num_features = len(feature_names)

    if num_importances != num_features:

        print(
            f"WARNING: {model_name} importance mismatch "
            f"({num_importances} importances vs "
            f"{num_features} feature names)"
        )

        feature_names = [
            f'Feature_{i}'
            for i in range(num_importances)
        ]



    feat_imp = pd.Series(
        importances,
        index=feature_names
    )

    feat_imp = feat_imp.sort_values(
        ascending=False
    )
    
    plt.figure(figsize=(10,6))
    
    top_features = feat_imp.head(top_n).sort_values()
    
    colors = plt.cm.viridis(
        np.linspace(0, 1, len(top_features))
    )
    
    top_features.plot(
        kind='barh',
        color=colors
    )
    
    plt.title(
        f'{model_name} Feature Importance'
    )
    
    plt.xlabel('Importance')
    
    plt.tight_layout()
    
    plt.show()
    

def plot_pca(
    X,
    y
):

    pca = PCA(
        n_components=2
    )

    X_pca = pca.fit_transform(X)

    plt.figure(figsize=(8,6))

    scatter = plt.scatter(
        X_pca[:,0],
        X_pca[:,1],
        c=y,
        alpha=0.6
    )

    plt.legend(
        handles=scatter.legend_elements()[0],
        labels=EMOTION_LABELS
    )

    plt.title(
        'PCA Projection'
    )

    plt.xlabel('Autonomic Activity')
    plt.ylabel('Movement and Blood Pulse Frequency')

    plt.show()


def plot_multiclass_roc(
    model,
    X_test,
    y_test,
    name
):

    y_score = model.predict_proba(
        X_test
    )

    y_test_bin = label_binarize(
        y_test,
        classes=[0,1,2,3]
    )

    plt.figure(figsize=(9,7))

    for i in range(4):

        fpr, tpr, _ = roc_curve(
            y_test_bin[:,i],
            y_score[:,i]
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            linewidth=2,
            label=f'{EMOTION_LABELS[i]} AUC={roc_auc:.2f}'
        )

    plt.plot(
        [0,1],
        [0,1],
        linestyle='--',
        color='black',
        alpha=0.5,
        label='Random Guess'
    )

    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')

    plt.title(f'{name} ROC Curves')

    plt.grid(
        True,
        linestyle='--',
        alpha=0.6
    )

    plt.xticks(
        np.arange(0, 1.1, 0.1)
    )

    plt.yticks(
        np.arange(0, 1.1, 0.1)
    )

    plt.xlim(-0.02,1.02)
    plt.ylim(-0.02,1.02)

    plt.legend()

    plt.tight_layout()

    plt.show()


def plot_signal_window(
    signal,
    label_name,
    signal_name='EDA'
):

    plt.figure(figsize=(12,4))

    time = np.arange(len(signal)) / 4

    plt.plot(time, signal)

    plt.title(
        f'{signal_name} Signal - {label_name}'
    )

    plt.xlabel('Time (Seconds)')
    plt.ylabel(signal_name + " (" + SIGNAL_UNITS[signal_name] + ")")

    plt.grid(
        True,
        linestyle='--',
        alpha=0.4
    )
        
    
    plt.show()


def plot_feature_distribution(
    feature_values,
    labels,
    feature_name
):

    df = pd.DataFrame({
        'Feature': feature_values,
        'Label': labels
    })

    plt.figure(figsize=(8,6))

    sns.boxplot(
        x='Label',
        y='Feature',
        data=df
    )

    plt.xticks(
        [0,1,2,3],
        EMOTION_LABELS
    )

    plt.title(
        f'{feature_name} Distribution'
    )

    plt.show()


def plot_correlation_heatmap(
    X,
    feature_names
):

    df = pd.DataFrame(
        X,
        columns=feature_names
    )

    corr = df.corr()

    plt.figure(figsize=(12,10))

    sns.heatmap(
        corr,
        cmap='coolwarm'
    )

    plt.title(
        'Feature Correlation Heatmap'
    )

    plt.show()

def plot_class_distribution(y):

    plt.figure(figsize=(8,6))

    ax = sns.countplot(x=y)

    colors = [
        'blue',     # Baseline
        'red',      # Stress
        'green',    # Amusement
        'purple'    # Meditation
    ]

    for i, bar in enumerate(ax.patches):

        bar.set_color(colors[i])

    ax.set_xticklabels([
        'Happy',
        'Sad',
        'Calm',
        'Angry'
    ])

    plt.title('Class Distribution')

    plt.xlabel('Emotion')

    plt.ylabel('Count')

    plt.grid(
        True,
        linestyle='--',
        alpha=0.4
    )
        

    plt.show()



def plot_time_series_with_labels(
    signal,
    labels,
    signal_name='EDA'
):

    plt.figure(figsize=(15,5))

    time = np.arange(len(signal)) / 4

    plt.plot(time, signal, linewidth=1, color='black')

    colors = {
        0: 'blue',
        1: 'red',
        2: 'green',
        3: 'purple'
    }
    label_names = EMOTION_LABELS

    current_label = labels[0]
    start_idx = 0

    for i in range(1, len(labels)):

        if labels[i] != current_label:

            plt.axvspan(
                start_idx /4,
                i/4,
                color=colors[current_label],
                alpha=0.2
            )


            start_idx = i
            current_label = labels[i]

    # last segment
    plt.axvspan(
        start_idx /4,
        len(labels) /4,
        color=colors[current_label],
        alpha=0.2
    )


    legend_patches = [
        mpatches.Patch(color=colors[i], alpha=0.2, label=label_names[i])
        for i in colors
    ]

    plt.legend(handles=legend_patches, loc='upper right')

    plt.title(f'{signal_name} Time Series with Emotion Labels')
    plt.xlabel('Time (Seconds)')
    plt.ylabel(signal_name + " (" + SIGNAL_UNITS[signal_name] + ")")

    plt.grid(
        True,
        linestyle='--',
        alpha=0.4
    )
        
    
    plt.show()


def plot_rolling_statistics(
    signal,
    window=100,
    signal_name='EDA'
):

    signal_series = pd.Series(signal)

    rolling_mean = signal_series.rolling(window).mean()
    rolling_std = signal_series.rolling(window).std()

    plt.figure(figsize=(15,6))

    time = np.arange(len(signal)) / 4

    plt.plot(
        time,
        signal,
        label='Original Signal',
        alpha=0.5
    )

    plt.plot(
        time,
        rolling_mean,
        label='Rolling Mean',
        linewidth=2
    )

    if(signal_name != 'TEMP'):
        plt.plot(
            time,
            rolling_std,
            label='Moving Std',
            linewidth=2
        )

    plt.title(
        f'{signal_name} Moving Statistics'
    )

    plt.xlabel('Time (Seconds)')
    plt.ylabel(signal_name + " (" + SIGNAL_UNITS[signal_name] + ")")

    plt.legend()

    plt.grid(
        True,
        linestyle='--',
        alpha=0.4
    )
        
    
    plt.show()

def plot_emotion_transitions(labels, sampling_rate = 4):

    plt.figure(figsize=(15,3))

    colors = {
        0: 'blue',
        1: 'red',
        2: 'green',
        3: 'purple'
    }

    time = np.arange(len(signal)) / sampling_rate
    
    # plot segmented lines
    current_label = labels[0]
    start_idx = 0
    

    for i in range(1, len(labels)):

        if labels[i] != current_label:

            segment_time = (
                np.arange(start_idx, i)
                / sampling_rate
            )
            
            plt.plot(
                segment_time,
                labels[start_idx:i],
                color=colors[current_label],
                linewidth=2
            )

            start_idx = i
            current_label = labels[i]
    
    # last segment
    segment_time = (
                np.arange(start_idx, i)
                / sampling_rate
            )
    plt.plot(
                segment_time,
                labels[start_idx:i],
                color=colors[current_label],
                linewidth=2
            )

    plt.yticks([0,1,2,3], EMOTION_LABELS)

    plt.title('Emotion Labels Over Time')
    plt.xlabel('Time (Seconds)')
    plt.ylabel('Emotion State')

 
    plt.show()


In [ ]:
# ============================================================
# RUNS ALL VISUALIZATIONS
# ============================================================

from pipelines.feature_extraction import build_amigos_features


feature_builders = {
        'Logistic Regression': build_logistic_features,
        'SVM': build_svm_features,
        'KNN': build_knn_features,
        'Random Forest': build_rf_features,
        'Gradient Boosting': build_gb_features,
        'XGBoost': build_xgb_features
    }


plt.rcParams.update({
    'font.size': 18,             # Increase text size
    'lines.linewidth': 1,        # Increase thickness of connecting lines
    'patch.linewidth': 1         # Increase thickness of node borders
})

print("\n===== RUNNING VISUALIZATIONS =====")




X_vis = all_X
y_vis = all_y


# ------------------------------------------------------------
# CLASS DISTRIBUTION
# ------------------------------------------------------------

plot_class_distribution(y_vis)


# ------------------------------------------------------------
# PCA
# ------------------------------------------------------------

plot_pca(
    X_vis,
    y_vis
)



# ------------------------------------------------------------
# CORRELATION HEATMAP
# ------------------------------------------------------------



X_rf = X_models['Random Forest']
names_rf = model_feature_names['Random Forest']

plot_correlation_heatmap(
    X_rf,
    names_rf
)


# ------------------------------------------------------------
# MODEL VISUALIZATIONS
# ------------------------------------------------------------


for name, model in models.items():

    print(f"\n===== VISUALIZING {name} =====")

    if "AMIGOS" in DATA_DIR:  
        builder = build_amigos_features
            
    else:
        builder = feature_builders[name]

    all_X_model = []
    all_y_model = []

    for signal, label in zip(signals, labels):

        Xi, yi, feature_names = create_model_windows(
            signal,
            label,
            builder
        )

        all_X_model.extend(Xi)
        all_y_model.extend(yi)

    all_X_model = np.array(all_X_model)
    all_y_model = np.array(all_y_model)

    X_train, X_test, y_train, y_test = train_test_split(
        all_X_model,
        all_y_model,
        test_size=0.2,
        random_state=42,
        stratify=all_y_model
    )

    # --------------------------------------------------------
    # ROC CURVE
    # --------------------------------------------------------

    if hasattr(model, "predict_proba"):

        plot_multiclass_roc(
            model,
            X_test,
            y_test,
            name
        )

    # --------------------------------------------------------
    # FEATURE IMPORTANCE
    # --------------------------------------------------------

    if name in [
        'Random Forest',
        'Gradient Boosting',
        'XGBoost'
    ]:

        plot_feature_importance(
            model,
            model_feature_names[name],
            name
        )


# ============================================================
# SIGNAL VISUALIZATIONS
# ============================================================

print("\n===== SIGNAL VISUALIZATIONS =====")

# Combine all subjects
sample_signal = np.concatenate(signals, axis=0)  
sample_labels = np.concatenate(labels)



if DATA_DIR == 'data/WESAD':
    length = 50000
else:
    length = 500000


# ------------------------------------------------------------
# SINGLE SIGNAL WINDOWS
# ------------------------------------------------------------

plot_signal_window(
    sample_signal[:length, 0],
    'Sample EDA',
    'EDA'
)

if DATA_DIR == 'data/WESAD':
    plot_signal_window(
        sample_signal[:length, 2],
        'Sample TEMP',
        'TEMP'
    )

plot_signal_window(
    sample_signal[:length, 1],
    'Sample BVP',
    'BVP'
)

if DATA_DIR == 'data/WESAD':
    plot_signal_window(
        sample_signal[:length, 3],
        'Sample ACC',
        'ACC'
    )

# ------------------------------------------------------------
# TIME SERIES + LABEL OVERLAY
# ------------------------------------------------------------

plot_time_series_with_labels(
    sample_signal[:length, 0],
    sample_labels[:length],
    'EDA'
)

if DATA_DIR == 'data/WESAD':
    plot_time_series_with_labels(
        sample_signal[:length, 2],
        sample_labels[:length],
        'TEMP'
    )

plot_time_series_with_labels(
    sample_signal[:length, 1],
    sample_labels[:length],
    'BVP'
)

if DATA_DIR == 'data/WESAD':
    plot_time_series_with_labels(
        sample_signal[:length, 3],
        sample_labels[:length],
        'ACC'
    )



# ------------------------------------------------------------
# ROLLING STATS
# ------------------------------------------------------------

plot_rolling_statistics(
    sample_signal[:length, 0],
    signal_name='EDA'
)

if DATA_DIR == 'data/WESAD':
    plot_rolling_statistics(
        sample_signal[:length, 2],
        signal_name='TEMP'
    )

plot_rolling_statistics(
    sample_signal[:length, 1],
    signal_name='BVP'
)

if DATA_DIR == 'data/WESAD':
    plot_rolling_statistics(
        sample_signal[:length, 3],
        signal_name='ACC'
    )
# ------------------------------------------------------------
# EMOTION TRANSITIONS
# ------------------------------------------------------------

plot_emotion_transitions(
    sample_labels[:length]
)

print("\n===== VISUALIZATION COMPLETE =====")

In [ ]:
#feature graphs

model_features = {

    'Logistic Regression': [
        'EDA Mean','EDA Std','EDA Min','EDA Max',
        'BVP Mean','BVP Std',
        'TEMP Mean','TEMP Std','TEMP Min','TEMP Max'
    ],

    'KNN': [
        'EDA Mean',
        'EDA Std',
        'EDA Diff Mean',
        'BVP Mean',
        'BVP Std',
        'BVP Diff Mean'
    ],

    'SVM': [
        'EDA Mean',
        'EDA Std',
        'EDA Min',
        'EDA Max',
        'EDA Median',
        'EDA Diff Mean',
        'EDA Diff Std',
        'EDA Peak Count',
        'EDA Avg Peak Height',
        'EDA Entropy',

        'BVP Mean',
        'BVP Std',
        'BVP Skew',
        'BVP Kurtosis',
        'BVP FFT Mean',
        'BVP FFT Std',
        'BVP FFT Max',
        'BVP Peak Count',
        'BVP Diff Mean',
        'BVP Diff Std',

        'TEMP Mean',
        'TEMP Std',
        'TEMP Min',
        'TEMP Max',
        'TEMP Diff Mean',
        'TEMP Diff Std'
    ],

    'Random Forest': feature_names[:32],

    'Gradient Boosting': feature_names[:32],

    'XGBoost': feature_names
}

summary = pd.DataFrame(
    index=model_features.keys(),
    columns=['EDA','BVP','TEMP','ACC']
)

for model, feats in model_features.items():

    summary.loc[model,'EDA'] = sum(
        f.startswith('EDA')
        for f in feats
    )

    summary.loc[model,'BVP'] = sum(
        f.startswith('BVP')
        for f in feats
    )

    summary.loc[model,'TEMP'] = sum(
        f.startswith('TEMP')
        for f in feats
    )

    summary.loc[model,'ACC'] = sum(
        f.startswith('ACC')
        for f in feats
    )

summary = summary.astype(int)

summary.plot(
    kind='bar',
    figsize=(10,6)
)

plt.title(
    'Feature Categories Used By Each Model'
)

plt.ylabel(
    'Number of Features'
)

plt.grid(alpha=0.3)

plt.tight_layout()

plt.show()


import matplotlib.pyplot as plt

feature_counts = {
    'Logistic Regression': 10,
    'KNN': 6,
    'SVM': 26,
    'Random Forest': 32,
    'Gradient Boosting': 32,
    'XGBoost': 36
}

plt.figure(figsize=(10, 5))


bars = plt.barh(
    list(feature_counts.keys()),
    list(feature_counts.values())
)


for bar in bars:
    plt.text(
        bar.get_width() + 0.5,               
        bar.get_y() + bar.get_height() / 2, 
        str(int(bar.get_width())),           
        va='center',                        
        ha='left'                            
    )


plt.xlabel(
    'Number of Features'
)

plt.title(
    'Feature Complexity by Model'
)


plt.grid(
    axis='x',
    alpha=0.3
)

plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()



all_features = sorted(
    set(
        f
        for feats in model_features.values()
        for f in feats
    )
)

heatmap_df = pd.DataFrame(
    0,
    index=all_features,
    columns=model_features.keys()
)

for model, feats in model_features.items():

    heatmap_df.loc[feats, model] = 1

plt.figure(figsize=(12,14))

sns.heatmap(
    heatmap_df,
    cmap='Blues',
    linewidths=.5,
    cbar=False
)

plt.title(
    'Features Used By Each Model'
)

plt.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Multiclass AUROC
# ------------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt
import numpy as np

def plot_model_comparison_auroc(
    models,
    X_models,
    y_models
):

    colors = {
        'Logistic Regression': 'blue',
        'SVM': 'red',
        'KNN': 'green',
        'Random Forest': 'orange',
        'Gradient Boosting': 'purple',
        'XGBoost': 'brown'
    }

    plt.figure(figsize=(10, 8))

    for name, model in models.items():

        X = np.array(X_models[name])
        y = np.array(y_models[name])

        _, X_test, _, y_test = train_test_split(
            X,
            y,
            test_size=0.2,
            random_state=42,
            stratify=y
        )

 
        if hasattr(model, "predict_proba"):

            y_score = model.predict_proba(X_test)

        elif hasattr(model, "decision_function"):

            y_score = model.decision_function(X_test)

            if y_score.ndim == 1:
                y_score = y_score.reshape(-1, 1)

        else:

            print(f"Skipping {name} (no probability scores)")
            continue

      
        y_test_bin = label_binarize(
            y_test,
            classes=np.unique(y)
        )

        fpr, tpr, _ = roc_curve(
            y_test_bin.ravel(),
            y_score.ravel()
        )

        roc_auc = auc(fpr, tpr)

        plt.plot(
            fpr,
            tpr,
            lw=2,
            color=colors.get(name, None),
            label=f"{name} (AUC = {roc_auc:.3f})"
        )

    plt.plot(
        [0, 1],
        [0, 1],
        'k--',
        lw=1
    )

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Multiclass AUROC Comparison")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()


plot_model_comparison_auroc(
    models,
    X_models,
    y_models
)

In [ ]:
# Logistic Regression graph - needs fixing can be ignored for now
import numpy as np
import matplotlib.pyplot as plt

class_names = {
    0: "Happy",
    1: "Sad",
    2: "Calm",
    3: "Angry"
}

class_colors = {
    0: "blue",
    1: "red",
    2: "green",
    3: "purple"
}


scaler = lr_pipe.named_steps['scaler']

eda_mean = scaler.mean_[eda_idx]
eda_std = scaler.scale_[eda_idx]



eda_scaled = np.linspace(
    np.percentile(X[:, eda_idx], 0),
    np.percentile(X[:, eda_idx], 95),
    300
)



eda_real = (
    eda_scaled * eda_std
) + eda_mean


X_curve = np.tile(
    happy,
    (len(eda_scaled), 1)
)

X_curve[:, eda_idx] = eda_scaled



probs = lr_pipe.predict_proba(X_curve)



plt.figure(figsize=(12,7))

for i, cls in enumerate(lr_pipe.classes_):

    plt.plot(
        eda_real,
        probs[:, i],
        linewidth=3,
        color=class_colors[cls],
        label=class_names[cls]
    )

plt.xlabel("EDA Maximum (µS)")
plt.ylabel("Predicted Probability")

plt.title(
    "Effect of EDA Maximum on Logistic Regression Predictions"
)

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#colors to be used for the SVM,KNN graphs
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

EMOTION_LABELS = {
    0: "Happy",
    1: "Sad",
    2: "Calm",
    3: "Angry"
}

emotion_cmap = ListedColormap([
    'blue',
    'red',
    'green',
    'purple'
])

legend_elements = [
    Patch(facecolor='blue', label="Happy"),
    Patch(facecolor='red', label="Sad"),
    Patch(facecolor='green', label="Calm"),
    Patch(facecolor='purple', label="Angry")
]

In [ ]:
#svm
from sklearn.decomposition import PCA
pca = PCA(n_components=2)

X_2d = pca.fit_transform(X_models['SVM'])


from sklearn.svm import SVC

svm_vis = SVC(
    kernel='rbf',
    probability=True
)

svm_vis.fit(X_2d, y_models['SVM'])
xx, yy = np.meshgrid(
    np.linspace(X_2d[:,0].min()-1, X_2d[:,0].max()+1, 300),
    np.linspace(X_2d[:,1].min()-1, X_2d[:,1].max()+1, 300)
)

grid = np.c_[xx.ravel(), yy.ravel()]

Z = svm_vis.predict(grid)
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10,8))

plt.contourf(xx, yy, Z, cmap= emotion_cmap, alpha=1)

scatter = plt.scatter(
    X_2d[:,0],
    X_2d[:,1],
    c=y_models['SVM'],
    cmap=emotion_cmap,
    edgecolors='black',
    s=40
)

#this highlights the points that contribute most but make the graph a little hard to read
#plt.scatter(
#    svm_vis.support_vectors_[:,0],
#    svm_vis.support_vectors_[:,1],
#    s=100,
#    facecolors='none',
#    edgecolors='red',
#    linewidths=.2,
#    label='Support Vectors'
#)

plt.title("SVM Decision Regions")
plt.xlabel("BVP FFT Max")
plt.ylabel("BVP FFT Mean")

plt.legend(
    handles=legend_elements,
    title="Emotion",
    loc="upper right"
)

plt.show()



In [ ]:
# KNN
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from sklearn.neighbors import KNeighborsClassifier



class_names = {
    0: "Happy",
    1: "Sad",
    2: "Calm",
    3: "Angry"
}

point_cmap = ListedColormap([
    "blue",      # Baseline
    "red",       # Stress
    "green",     # Amusement
    "purple"     # Meditation
])



knn_vis = KNeighborsClassifier(
    n_neighbors=5
)

knn_vis.fit(
    X_2d,
    y_models['KNN']
)


Z = knn_vis.predict_proba(grid)


fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 12)
)

axes = axes.ravel()


for plot_idx, cls in enumerate(knn_vis.classes_):

    ax = axes[plot_idx]

    class_idx = list(knn_vis.classes_).index(cls)

    Z_class = Z[:, class_idx]
    Z_class = Z_class.reshape(xx.shape)

    class_colors = {
        0: "Blues",
        1: "Reds",
        2: "Greens",
        3: "Purples"
    }
    
    contour = ax.contourf(
        xx,
        yy,
        Z_class,
        levels=30,
        cmap=class_colors[cls],
        alpha=0.8
    )

    scatter = ax.scatter(
        X_2d[:, 0],
        X_2d[:, 1],
        c=y_models['KNN'],
        cmap=point_cmap,
        edgecolors='k',
        s=25
    )

    cbar = fig.colorbar(
        contour,
        ax=ax
    )

    cbar.set_label(
        f"{class_names[cls]} Probability"
    )

    ax.set_title(
        f"KNN {class_names[cls]} Probability Surface"
    )

    ax.legend(
        handles=legend_elements,
        title="Emotion",
        loc="upper right"
    )

    ax.set_xlabel("BVP FFT Max")
    ax.set_ylabel("BVP FFT Mean")



plt.suptitle(
    f"Probability of Class using KNN",
    fontsize=18
)

plt.tight_layout()

plt.show()

In [ ]:
#xgboost
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from xgboost import plot_tree 


xgb_model = models['XGBoost']
X_xgb = np.array(X_models['XGBoost'])
y_xgb = np.array(y_models['XGBoost'])


X_small = X_xgb[:, top_idx]


X_small_df = pd.DataFrame(
    X_small, 
    columns=top_feature_names
)


xgb_demo = XGBClassifier(
    n_estimators=1,
    max_depth=2,
    learning_rate=1.0,
    objective='multi:softprob',
    num_class=4,
    random_state=42
)
xgb_demo.fit(
    X_small_df, 
    y_xgb
)


plt.rcParams.update({
    'font.size': 14,             
    'lines.linewidth': 3.5,       
    'patch.linewidth': 2.5        
})


fig, ax = plt.subplots(figsize=(25, 12))


plot_tree(
    xgb_demo, 
    num_trees=0, 
    rankdir='TB',
    ax=ax
)


plt.title(
    "XGBoost Explanation Tree\n(Top 3 Features)",
    fontsize=22,
    weight='bold',
    pad=20
)


plt.savefig(
    'xgboost_explanation_tree_highres.png', 
    dpi=300, 
    bbox_inches='tight'
)


plt.show()



sample_idx = 100
sample = X_small_df.iloc[[sample_idx]]


pred = xgb_demo.predict(sample)[0] 

emotion_names = {
    0: "Happy",
    1: "Sad",
    2: "Calm",
    3: "Angry"
}

print("\nSample Values:\n")
for col in sample.columns:
    print(f"{col}: {sample.iloc[0][col]:.3f}")

print("\nPredicted Emotion:")
print(emotion_names[pred])




In [ ]:
#gradient boosting
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

gb = models['Gradient Boosting']

X = np.array(X_models['Gradient Boosting'])
y = np.array(y_models['Gradient Boosting'])

_, X_test_gb, _, y_test_gb = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

train_scores = []

for pred in gb.staged_predict(X_test_gb):

    acc = accuracy_score(
        y_test_gb,
        pred
    )

    train_scores.append(acc)

plt.figure(figsize=(10,6))

plt.plot(
    range(1, len(train_scores)+1),
    train_scores,
    linewidth=3
)

plt.xlabel("Boosting Stage")
plt.ylabel("Accuracy")
plt.title("Gradient Boosting Learning Progression")

plt.grid(True)
plt.show()


train_loss = gb.train_score_

plt.figure(figsize=(10,6))

plt.plot(train_loss, linewidth=3)

plt.xlabel("Boosting Stage")
plt.ylabel("Training Loss")

plt.title("Residual Error Reduced by Boosting")

plt.grid(True)

plt.show()

In [ ]:
#rf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

rf = models['Random Forest']
X_rf = np.array(X_models['Random Forest'])
y_rf = np.array(y_models['Random Forest'])


explanation_tree = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)
explanation_tree.fit(
    X_small, 
    y_rf
)


class_names = [
    "Happy",
    "Sad",
    "Calm",
    "Angry"
]

plt.rcParams.update({
    'font.size': 18,             # Increase text size
    'lines.linewidth': 2,        # Increase thickness of connecting lines
    'patch.linewidth': 2         # Increase thickness of node borders
})

plt.figure(figsize=(25,12))


annotations = plot_tree(
    explanation_tree,
    feature_names=top_feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=18
)


for text_obj in annotations:
    old_text = text_obj.get_text()
    lines = old_text.split('\n')
    
    
    cleaned_lines = [
        line for line in lines 
        if not any(word in line for word in ['gini', 'samples', 'value'])
    ]
    
    
    text_obj.set_text('\n'.join(cleaned_lines))

plt.title(
    "Random Forest Explanation Tree"
)
plt.show()


In [ ]:
#use ollama as our LLM component, this is an example of how I think we should/could use LLMs

from langchain_ollama import ChatOllama

llm = ChatOllama(model='llama3')

prompt = f'''
Analyze these WESAD emotion detection results:

{results}

Explain:
1. Which model performed best

2. What might a subject who is really stressed have as physiological data?

3. What emotion would the following sample be: 
    "eda_mean": 5.3,
    "eda_std": 1.1,
    "bvp_mean": 72.5,
    "bvp_std": 9.4,
    "temp_mean": 32.0,
    "temp_std": 0.3,
    "acc_mean": 0.9,
    "acc_std": 0.5,
    "peak_count": 9,
    "signal_entropy": 3.4


3. What emotion would the following sample be: 
    "eda_mean": 1.8,
    "eda_std": 0.2,
    "bvp_mean": 60.2,
    "bvp_std": 2.8,
    "temp_mean": 33.1,
    "temp_std": 0.1,
    "acc_mean": 0.1,
    "acc_std": 0.05,
    "peak_count": 1,
    "signal_entropy": 0.9

'''

response = llm.invoke(prompt)

print(response.content)